# 01 · 共享 global R1 的真实层 G1

逐层采集 q/up 输入侧和 down 输出侧；所有面板共享同一个 Q。完整 reduction groups 与输出 hidden 轴保留。默认只执行 preflight，不下载或启动模型。

GitHub / VS Code 工作流先运行 `00_Colab_GitHub_Workspace.ipynb`；详见 [操作说明](../docs/COLAB_VSCODE.md)。已有 ZIP 入口仍可用。

## 路径与环境配置

GitHub / VS Code 首先运行 `00_Colab_GitHub_Workspace.ipynb`；本 notebook 会识别 `/content/CellQuant` 并读取运行时路径配置。
已有 ZIP 仍可设置 `KIT_ARCHIVE` 和 `EXTRACT_ARCHIVE=True` 解压，或直接指定 `PROJECT_ROOT`。
默认 smoke / manifest-only，不下载模型、不启动全量 sweep。基础依赖缺失时才设置 `INSTALL_BASE_DEPENDENCIES=True`，不替换宿主 torch。
真实模型授权通过环境变量或隐藏输入提供，不能写入 notebook 输出或 Git；阶段结束用 workspace backup 保存到 Drive。


In [1]:
from pathlib import Path
import os, sys, json

# Colab: Git clone 到 /content/CellQuant；也兼容完整 ZIP 解压目录。
# AutoDL: 建议 /root/autodl-tmp/cellquant_research_kit。
# 留 None 时依次查找当前目录、父目录和上述两个位置。
PROJECT_ROOT = os.environ.get("CELLQUANT_PROJECT_ROOT") or ("/content/CellQuant" if Path("/content/CellQuant/pyproject.toml").is_file() else None)
RUN_ROOT = os.environ.get("CELLQUANT_RUN_ROOT") or None
KIT_ARCHIVE = None  # 例：/content/drive/MyDrive/cellquant_research_kit.zip
EXTRACT_ARCHIVE = False  # 首次运行且 ZIP 已上传时才改 True
ARCHIVE_DESTINATION = "/content/cellquant_workspace" if Path("/content").exists() else str(Path.cwd().parent / "cellquant_workspace")
MOUNT_DRIVE = Path("/content").is_dir()  # Colab 默认挂载，本地/AutoDL 默认跳过
DURABLE_ROOT = None  # 例：/content/drive/MyDrive/cellquant_runs；不存权重或 token
INSTALL_BASE_DEPENDENCIES = True
SEED = 20261115

## 挂载 Google Drive

运行下方单元，按 Google 弹窗完成登录授权，然后继续运行后续单元。
此命令同时用于浏览器 Colab 和 VS Code 的 Colab 内核；已挂载时会复用，非 Colab 环境默认跳过。
`CellQuant` 项目文件夹会在后续初始化中自动创建。


In [2]:
# 在 Colab（浏览器或 VS Code 内核）中直接挂载 Google Drive。
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive 挂载尚未完成，请完成授权后重跑本单元。")
    print("Google Drive 已挂载：/content/drive/MyDrive")
else:
    print("非 Colab 环境或 MOUNT_DRIVE=False：跳过云盘挂载。")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive 已挂载：/content/drive/MyDrive


In [3]:
archive_roots = []
if EXTRACT_ARCHIVE:
    import zipfile, hashlib
    if not KIT_ARCHIVE or not Path(KIT_ARCHIVE).is_file():
        raise FileNotFoundError("先上传完整 ZIP 并设置 KIT_ARCHIVE")
    archive_path = Path(KIT_ARCHIVE)
    destination = Path(ARCHIVE_DESTINATION).resolve()
    digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
    receipt = destination / "archive_source_sha256.txt"
    if destination.exists() and any(destination.iterdir()):
        if not receipt.is_file() or receipt.read_text().strip() != digest:
            raise FileExistsError("目标目录已有不同内容；请设置新的 ARCHIVE_DESTINATION，避免覆盖修改")
        print("Same archive already extracted; preserving existing project edits.")
    else:
        with zipfile.ZipFile(archive_path) as archive:
            infos = archive.infolist()
            if sum(info.file_size for info in infos) > 2 * 2**30:
                raise ValueError("项目包解压体积超过 2GiB；请勿把基础权重放进代码包")
            for info in infos:
                target = (destination / info.filename).resolve()
                if not target.is_relative_to(destination) or (info.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError("Unsafe archive member")
            destination.mkdir(parents=True, exist_ok=True)
            archive.extractall(destination)
        receipt.write_text(digest + "\n")
    archive_roots = [destination, destination / "cellquant_research_kit"]

candidates = ([Path(PROJECT_ROOT)] if PROJECT_ROOT else []) + archive_roots + [
    Path.cwd(), Path.cwd().parent,
    Path("/content/cellquant_research_kit"),
    Path("/root/autodl-tmp/cellquant_research_kit"),
]
roots = [p.resolve() for p in candidates
         if (p / "pyproject.toml").is_file() and (p / "src/cellquant").is_dir()]
if not roots:
    raise FileNotFoundError("请先解压完整项目包，并设置 PROJECT_ROOT；只上传 notebook 不足以运行。")
PROJECT_ROOT = roots[0]
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ["CELLQUANT_PROJECT_ROOT"] = str(PROJECT_ROOT)

if INSTALL_BASE_DEPENDENCIES:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r",
                           str(PROJECT_ROOT / "requirements/base.txt")])

from cellquant.runtime import (atomic_json, environment_report, resolve_paths,
                              build_manifest, verify_manifest, copy_completed_run)
PATHS = resolve_paths(PROJECT_ROOT, RUN_ROOT, DURABLE_ROOT)
print("Project:", PATHS["project"])
print("Runs:", PATHS["runs"])

os.chdir(PROJECT_ROOT)
os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ.setdefault("HF_HOME", str(PROJECT_ROOT / ".cache/huggingface"))
os.environ.setdefault("PYTHONUTF8", "1")

# Separate notebook kernels reuse the same local runtime path configuration.
workspace_record = PROJECT_ROOT / ".cache/colab_workspace.json"
if workspace_record.is_file():
    from cellquant.workspace import configure_workspace
    saved = json.loads(workspace_record.read_text(encoding="utf-8"))
    configure_workspace(PROJECT_ROOT, saved["environment"]["CELLQUANT_DRIVE_ROOT"])


Project: /content/CellQuant
Runs: /content/CellQuant/runs


先按 `docs/COLAB_GPU_FROM_SCRATCH.md` 完成新服务器初始化。workspace 的 GPU 驻留检查应为 `PASS_ENGINEERING_ONLY`，真实输入准备输出为 `runs/prepared/g1_gpu/config.json`。模型直接加载到 GPU，全部种子的面板采集完成后释放基础模型；面板、共享 Q、五种求解/对照保留在 GPU。显存不足明确 RESOURCE_BLOCKED，不回退 CPU。文件读写、tokenizer、校验和、日志与调度仍使用 CPU。


先诊断事件求解器时，在下一单元设置 `RUN_REAL_MODEL=True`、`RUN_PAIR_PROFILE=True`。它使用完整真实面板、原角域和原时间预算，只运行第一个种子的第一个坐标对，检查固定尺度搜索与 CellQuant；不会生成 G1 证据。查看 `pair_profile.json` 和日志中的 `search_phase`、事件数与已扫描区间数。
只有诊断明确可运行后，再将 `RUN_PAIR_PROFILE=False` 运行正式 G1。它不会把旧代码下的已完成基线混入新运行；已有结果应保留。


In [4]:
from cellquant.experiment import run_experiment
from cellquant.runtime import stream_command, read_json
from cellquant.gates import evaluate_file
OUT = PATHS["runs"] / "01_pilot"
OUT.mkdir(parents=True, exist_ok=True)
RUN_REAL_MODEL = True
PILOT_CONFIG_PATH = Path(os.environ.get("CELLQUANT_PILOT_CONFIG", PROJECT_ROOT / "runs/prepared/g1_gpu/config.json"))
if not PILOT_CONFIG_PATH.is_absolute():
    PILOT_CONFIG_PATH = PROJECT_ROOT / PILOT_CONFIG_PATH
if not PILOT_CONFIG_PATH.is_file():
    if RUN_REAL_MODEL:
        raise FileNotFoundError(
            f"真实 G1 的准备配置不存在：{PILOT_CONFIG_PATH}\n"
            "请在 00_Colab_GitHub_Workspace.ipynb 设置 PREPARE_G1_INPUTS=True，"
            "运行“准备真实 G1 输入”单元，完成数据准备、权重校验和 Drive 备份后再重跑本单元。\n"
            "HF 登录单元下载的 config.json 只是模型说明文件，不是本实验的准备配置。"
        )
    print("准备配置未找到；仅以草案进行预检查，不执行模型。")
    PILOT_CONFIG_PATH = PROJECT_ROOT / "configs/colab_pilot.json"
RUN_PAIR_PROFILE = True  # True：仅诊断第一个坐标对，不生成 G1 成绩
PILOT_RUN_DIR = None
if RUN_REAL_MODEL:
    selected_config = read_json(PILOT_CONFIG_PATH)
    if (selected_config.get("runtime", {}).get("cpu_offload") is not False
            or not selected_config.get("runtime", {}).get("model_device", "").startswith("cuda")
            or selected_config.get("solver", {}).get("backend") != "torch"
            or not selected_config.get("solver", {}).get("device", "").startswith("cuda")):
        raise RuntimeError("这是旧的 CPU 配置。请在 workspace 准备新的 runs/prepared/g1_gpu/config.json；不要覆盖旧实验或只改 RUN_REAL_MODEL。")
print("实际使用的 G1 配置：", PILOT_CONFIG_PATH)
preflight = run_experiment(PILOT_CONFIG_PATH, project_root=PROJECT_ROOT, preflight_only=True)
print(preflight["readiness"])


实际使用的 G1 配置： /content/CellQuant/runs/prepared/g1_gpu/config.json
{'status': 'READY', 'missing': [], 'environment': {'schema_version': '1.0', 'created_at_utc': '2026-11-17T10:20:16.967857+00:00', 'python': '3.12.13', 'executable': '/usr/bin/python3', 'platform': 'Linux-6.6.159+-x86_64-with-glibc2.35', 'machine': 'x86_64', 'packages': {'numpy': '2.0.2', 'scipy': '1.16.3', 'torch': '2.11.0+cu128', 'transformers': '4.44.2', 'datasets': '2.20.0', 'accelerate': '0.34.2', 'safetensors': '0.8.0', 'nbformat': '5.10.4', 'nbclient': '0.10.4', 'fast-hadamard-transform': '1.1.0', 'flash-attn': None, 'triton': '3.6.0', 'lm_eval': None, 'tokenizers': '0.19.1', 'huggingface-hub': '0.36.2', 'pyarrow': '18.1.0', 'psutil': '5.9.5'}, 'cuda': {'available': True, 'torch_cuda_build': '12.8', 'status': 'DEVICE_DETECTED', 'devices': [{'index': 0, 'name': 'NVIDIA L4', 'total_gib': 22.0343017578125, 'compute_capability': [8, 9]}], 'bf16_supported': True, 'nvidia_smi': 'NVIDIA L4, 580.82.07, 23034 MiB, 8.9'}, 'co

In [5]:
import os, json, subprocess, torch
cfg = json.loads(PILOT_CONFIG_PATH.read_text())
dev = torch.device(cfg['runtime'].get('model_device', 'cuda:0'))
free, total = torch.cuda.mem_get_info(dev)
fraction = cfg['runtime'].get('max_memory_fraction', 0.85)
print({'PID': os.getpid(), 'GPU': torch.cuda.get_device_name(dev),
       'total_GiB': total / 2**30, 'free_GiB': free / 2**30,
       'budget_GiB': total / 2**30 * fraction,
       'allocated_this_kernel_GiB': torch.cuda.memory_allocated(dev) / 2**30,
       'reserved_this_kernel_GiB': torch.cuda.memory_reserved(dev) / 2**30,
       'dtype': cfg['runtime']['dtype']})
subprocess.run(['nvidia-smi'], check=False)

{'PID': 43577, 'GPU': 'NVIDIA L4', 'total_GiB': 22.0343017578125, 'free_GiB': 21.658203125, 'budget_GiB': 18.729156494140625, 'allocated_this_kernel_GiB': 0.0, 'reserved_this_kernel_GiB': 0.0, 'dtype': 'bfloat16'}


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [6]:
if RUN_REAL_MODEL:
    if preflight["readiness"]["status"] != "READY":
        reasons = "\n".join("- " + reason for reason in preflight["readiness"]["missing"])
        raise RuntimeError(
            f"真实 G1 预检查仍为 BLOCKED。配置：{PILOT_CONFIG_PATH}\n"
            f"{reasons}\n"
            "先解决这些前置项并重跑上一个预检查单元；RUN_REAL_MODEL 不能代替科学证据。"
        )
    import uuid
    mode = "pair_profile" if globals().get("RUN_PAIR_PROFILE", False) else "pilot"
    PILOT_LOG = OUT / f"{mode}_{uuid.uuid4().hex[:12]}.log"
    PILOT_RESULT_INDEX = PILOT_LOG.with_suffix(".result_index.json")
    print("Pilot log:", PILOT_LOG, flush=True)
    command = [sys.executable, "-u", "-m", "cellquant.experiment", "--config", str(PILOT_CONFIG_PATH), "--result-index", str(PILOT_RESULT_INDEX)]
    if globals().get("RUN_PAIR_PROFILE", False):
        command.append("--profile-pair")
    process = stream_command(command,
                             PILOT_LOG, cwd=PROJECT_ROOT, timeout_s=5*3600)
    if PILOT_RESULT_INDEX.is_file():
        PILOT_RUN_DIR = Path(read_json(PILOT_RESULT_INDEX)["run_dir"])
        print("Actual run:", PILOT_RUN_DIR)
    if process["returncode"] != 0:
        detail = {}
        if PILOT_RUN_DIR is not None and (PILOT_RUN_DIR / "result.json").is_file():
            detail = read_json(PILOT_RUN_DIR / "result.json")
            if not detail.get("reason"):
                from cellquant.pilot import summarize_pilot_results
                paths = list(PILOT_RUN_DIR.glob("seed_*/*/pilot_result.json"))
                if paths:
                    detail.update(summarize_pilot_results(paths))
        reason = detail.get("reason") or "未生成具体失败原因，请查看日志"
        raise RuntimeError(f"{detail.get('status', 'INCOMPLETE')}: {reason}\n运行目录：{PILOT_RUN_DIR}\n日志：{PILOT_LOG}")
else:
    print("NOT_RUN: real-model execution switch is false")


Pilot log: /content/CellQuant/runs/01_pilot/pair_profile_8eeaad42f9ac.log
[CellQuant] run_dir=/content/CellQuant/runs/experiments/2026-11-17T102020.290922_0000_29bbe398e6 pid=43677
[CellQuant] {"event": "progress", "status": "RUNNING", "phase": "initialization", "details": {}, "elapsed_seconds": 0.0, "phase_elapsed_seconds": 0.0, "process_rss_gib": 0.638, "system_available_gib": 47.966, "process_cpu_seconds": 4.81, "progress_sequence": 1, "updated_at_utc": "2026-11-17T10:20:22.171706+00:00", "cuda_allocated_gib": 0.0, "cuda_reserved_gib": 0.0}
[CellQuant] {"event": "progress", "status": "RUNNING", "phase": "load", "details": {}, "elapsed_seconds": 0.02, "phase_elapsed_seconds": 0.0, "process_rss_gib": 0.64, "system_available_gib": 47.964, "process_cpu_seconds": 4.83, "progress_sequence": 2, "updated_at_utc": "2026-11-17T10:20:22.193521+00:00", "cuda_allocated_gib": 0.0, "cuda_reserved_gib": 0.0}
Fetching 3 files: 100%|██████████| 3/3 [00:00<00:00, 45756.04it/s]
[CellQuant] {"event": "h

RuntimeError: RESOURCE_BLOCKED: fixed_scale_search_matched: Pair wall-clock budget exceeded: 300s; cellquant: Pair wall-clock budget exceeded: 300s
运行目录：/content/CellQuant/runs/experiments/2026-11-17T102020.290922_0000_29bbe398e6
日志：/content/CellQuant/runs/01_pilot/pair_profile_8eeaad42f9ac.log

In [ ]:
from pathlib import Path
import json
run = Path('/content/CellQuant/runs/experiments/2026-11-17T081006.337925_0000_a3db20ca9b')
print('solver config:', json.loads((run / 'config.json').read_text())['solver'])
for p in sorted(run.glob('seed_*/*/pilot_result.json')):
    r = json.loads(p.read_text())
    s = r.get('solver', {})
    print(json.dumps({'seed': r.get('seed'), 'method': r.get('method'),
                      'status': r.get('status'), 'reason': s.get('reason'),
                      'seconds': s.get('seconds'),
                      'completed_pairs': len(s.get('log', []))}, ensure_ascii=False))

solver config: {'backend': 'torch', 'compute_dtype': 'float64', 'device': 'cuda:0', 'event_limit_per_pair': 200000, 'fallback_policy': 'report_resource_blocked_do_not_truncate_groups', 'initialization': 'torch_cuda_hadamard_v1', 'pair_budget': 128, 'root_solver': 'torch_complex_shifted_qr_v1', 'seconds_per_pair': 300, 'theta_limit': 0.7853981633974483, 'wall_seconds': 3600}
{"seed": 11, "method": "angle_grid_matched", "status": "COMPLETE", "reason": null, "seconds": 232.53729189600017, "completed_pairs": 128}
{"seed": 11, "method": "cellquant", "status": "RESOURCE_BLOCKED", "reason": "Pair wall-clock budget exceeded: 300s", "seconds": 307.82472268900074, "completed_pairs": 0}
{"seed": 11, "method": "fixed_scale_search_matched", "status": "RESOURCE_BLOCKED", "reason": "Pair wall-clock budget exceeded: 300s", "seconds": 307.8021749310001, "completed_pairs": 0}
{"seed": 11, "method": "givens_ste_matched", "status": "COMPLETE", "reason": null, "seconds": 514.758685882, "completed_pairs": 1

In [ ]:
import psutil, time
from pathlib import Path

assert Path("/content/CellQuant").is_dir(), "请在原 Colab 服务器运行"

for sample in range(2):
    mem = psutil.virtual_memory()
    print(f"\n采样 {sample + 1}：剩余 RAM {mem.available / 2**30:.1f} GiB")
    jobs = [
        p for p in psutil.process_iter(["cmdline"])
        if "cellquant.experiment" in (p.info["cmdline"] or [])
    ]
    print("实验进程数量：", len(jobs))
    for p in jobs:
        try:
            cpu = p.cpu_percent(interval=1)
            t = p.cpu_times()
            print("PID:", p.pid, "CPU%:", cpu,
                  "累计CPU秒:", round(t.user + t.system, 1),
                  "RSS GiB:", round(p.memory_info().rss / 2**30, 1))
        except psutil.Error as e:
            print(p.pid, type(e).__name__)
    if sample == 0:
        time.sleep(10)

### 查看运行进度（只读）

新版实际进程每 30 秒输出资源心跳，阶段变化时输出层号、seed、方法、面板或坐标对；`heartbeat` 只说明进程存活，不表示算法有进展。模型与采集面板在 GPU，求解数值后端为 torch CUDA；观察 `model_placement.json` 和心跳中的 `cuda_allocated_gib`。采集结束会释放基础模型以腾出显存，面板继续留在 GPU。Python 调度与同步可能使利用率波动，不能把“100% 利用率”当正确性标准。每个方法的预算也不包含全部准备及 held-out 计算，不能把预算当作完成时间承诺。

如果上方单元仍在执行，同一 kernel 中的下方单元会排队。请在能访问同一个 `PROJECT_ROOT` 的独立终端运行 `python -m cellquant.diagnostics --project-root /content/CellQuant`，或将下方只读单元复制到同一服务器的独立 kernel；不需要另建 Colab 服务器。隔 30 秒检查两次 CPU 累计时间、进度序号、剩余 RAM 与进程状态。旧代码的 `fusion` 状态可能已经过时，诊断会明确提示；单次 CPU 为零不能证明死锁。

不要重复启动模型进程。正在运行的旧进程不会自动得到新版日志；先诊断并保留旧日志，待作业结束或主动停止后再 pull、重启 kernel，以新 run 运行，不能把新代码混接进旧实验。


In [ ]:
from cellquant.diagnostics import inspect_runs
print(json.dumps(inspect_runs(PROJECT_ROOT), ensure_ascii=False, indent=2))


真实作业写入唯一 `runs/experiments/<run_id>`。把该目录中的 G1_evidence.json 设为下方路径；G1 从原始测量重算。运行标志不会更改门槛。超事件/时间预算记录 RESOURCE_BLOCKED；未完成控制组不产生性能结论。

In [ ]:
if globals().get("RUN_PAIR_PROFILE", False):
    print("NOT_RUN: 当前仅诊断一个坐标对；不生成或评估 G1 成绩。")
else:
    default_evidence = PILOT_RUN_DIR / "G1_evidence.json" if globals().get("PILOT_RUN_DIR") else PROJECT_ROOT / "runs/gates/G1_evidence.json"
    G1_EVIDENCE = Path(os.environ.get("CELLQUANT_G1_EVIDENCE", default_evidence))
    evidence_path = G1_EVIDENCE if G1_EVIDENCE.is_file() else PROJECT_ROOT / "examples/G1_evidence_NOT_RUN.json"
    gate = evaluate_file("G1", evidence_path)
    atomic_json(OUT / "G1.json", gate)
    print(gate["status"], gate["missing_fields"])
